In [1]:
!pip install gradio pandas numpy tensorflow joblib kafka-python plotly shap

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 326.3/326.3 kB 5.1 MB/s eta 0:00:00


In [2]:
!apt-get install nodejs npm -y
!npm install -g localtunnel

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  gyp javascript-common libc-ares2 libjs-events libjs-highlight.js
  libjs-inherits libjs-is-typedarray libjs-psl libjs-source-map
  libjs-sprintf-js libjs-typedarray-to-buffer libnode-dev libnode72
  libnotify-bin libnotify4 libuv1-dev node-abab node-abbrev node-agent-base
  node-ansi-regex node-ansi-styles node-ansistyles node-aproba node-archy
  node-are-we-there-yet node-argparse node-arrify node-asap node-asynckit
  node-balanced-match node-brace-expansion node-builtins node-cacache
  node-chalk node-chownr node-clean-yaml-object node-cli-table node-clone
  node-color-convert node-color-name node-colors node-columnify
  node-combined-stream node-commander node-console-control-strings
  node-copy-concurrently node-core-util-is node-coveralls node-cssom
  node-cssstyle node-debug node-decompress-response node-defaults
  node-delayed-st

In [ ]:
# !curl ipv4.icanhazip.com  # Copy this IP address
# !npx localtunnel --port 7860

In [3]:
from kafka import KafkaProducer
import json

In [6]:
import gradio as gr
import pandas as pd
import numpy as np
import tensorflow as tf
import joblib
import json
import shap
import os
import plotly.graph_objects as go
from google.colab import drive
from kafka import KafkaProducer
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import nest_asyncio
nest_asyncio.apply()

# ==============================
# 1. INITIALIZATION
# ==============================

drive.mount('/content/drive')
BASE_PATH = "/content/drive/MyDrive/Colab Notebooks"

model          = tf.keras.models.load_model(f"{BASE_PATH}/mlp_model.keras")
scaler         = joblib.load(f"{BASE_PATH}/scaler.pkl")
gatekeeper_pkg = joblib.load(f"{BASE_PATH}/gatekeeper_network.pkl")

mlp_threshold  = float(np.load(f"{BASE_PATH}/threshold.npy"))
print(f"MLP threshold loaded: {mlp_threshold:.4f}")

trained_features = gatekeeper_pkg['features']

producer = KafkaProducer(
    bootstrap_servers="kafka-24f52108-loginvestigation.i.aivencloud.com:27515",
    security_protocol="SSL",
    ssl_cafile=f"{BASE_PATH}/ca_b.pem",
    ssl_certfile=f"{BASE_PATH}/service_b.cert",
    ssl_keyfile=f"{BASE_PATH}/service_b.key",
    value_serializer=lambda v: json.dumps(v, default=str).encode("utf-8"),
    batch_size=131072,
    linger_ms=100,
    compression_type="gzip",
    acks=1
)

mlp_features = [
    "IN_BYTES","OUT_BYTES","IN_PKTS","OUT_PKTS",
    "FLOW_DURATION_MILLISECONDS",
    "LONGEST_FLOW_PKT","SHORTEST_FLOW_PKT",
    "TCP_FLAGS","CLIENT_TCP_FLAGS","SERVER_TCP_FLAGS",
    "RETRANSMITTED_IN_PKTS","RETRANSMITTED_OUT_PKTS",
    "MIN_TTL","MAX_TTL","PROTOCOL","L7_PROTO",
    "BYTE_RATIO","PKT_RATIO",
    "BYTES_PER_PKT_IN","BYTES_PER_PKT_OUT",
    "TOTAL_BYTES","TOTAL_PKTS",
    "BYTES_PER_MS","PKTS_PER_MS",
    "PKT_SIZE_RATIO",
    "RETRANS_RATIO_IN","RETRANS_RATIO_OUT",
    "TTL_DIFF"
]

# ==============================
# 2. PREPROCESSING PIPELINE
# ==============================

def preprocess_logs(df):
    df  = df.copy()
    df  = df.fillna(0)
    eps = 1e-9

    df["FLOW_START_MILLISECONDS"] = pd.to_datetime(
        df["FLOW_START_MILLISECONDS"], unit="ms", errors="coerce"
    )
    df["START_HOUR"] = df["FLOW_START_MILLISECONDS"].dt.hour
    df["hour_sin"]   = np.sin(2 * np.pi * df["START_HOUR"] / 24)
    df["hour_cos"]   = np.cos(2 * np.pi * df["START_HOUR"] / 24)

    df["BYTE_RATIO"]        = df["IN_BYTES"]  / (df["OUT_BYTES"]  + eps)
    df["PKT_RATIO"]         = df["IN_PKTS"]   / (df["OUT_PKTS"]   + eps)
    df["BYTES_PER_PKT_IN"]  = df["IN_BYTES"]  / (df["IN_PKTS"]    + eps)
    df["BYTES_PER_PKT_OUT"] = df["OUT_BYTES"] / (df["OUT_PKTS"]   + eps)
    df["TOTAL_BYTES"]       = df["IN_BYTES"]  +  df["OUT_BYTES"]
    df["TOTAL_PKTS"]        = df["IN_PKTS"]   +  df["OUT_PKTS"]
    df["BYTES_PER_MS"]      = df["TOTAL_BYTES"] / (df["FLOW_DURATION_MILLISECONDS"] + eps)
    df["PKTS_PER_MS"]       = df["TOTAL_PKTS"]  / (df["FLOW_DURATION_MILLISECONDS"] + eps)
    df["PKT_SIZE_RATIO"]    = df["LONGEST_FLOW_PKT"] / (df["SHORTEST_FLOW_PKT"] + eps)
    df["RETRANS_RATIO_IN"]  = df["RETRANSMITTED_IN_PKTS"]  / (df["IN_PKTS"]  + eps)
    df["RETRANS_RATIO_OUT"] = df["RETRANSMITTED_OUT_PKTS"] / (df["OUT_PKTS"] + eps)
    df["TTL_DIFF"]          = df["MAX_TTL"] - df["MIN_TTL"]

    df["BYTE_RATIO"]     = df["BYTE_RATIO"].clip(0, 500)
    df["PKT_RATIO"]      = df["PKT_RATIO"].clip(0, 500)
    df["PKT_SIZE_RATIO"] = df["PKT_SIZE_RATIO"].clip(0, 200)

    df = df.drop(columns=[
        "FLOW_START_MILLISECONDS", "FLOW_END_MILLISECONDS", "START_HOUR"
    ], errors="ignore")

    return df

# ==============================
# 3. GAUGE CHART
# ==============================

def create_gauge(score):
    fig = go.Figure(go.Indicator(
        mode="gauge+number",
        value=score,
        title={'text': "Threat Level"},
        gauge={
            'axis': {'range': [0, 100]},
            'bar': {'color': "#ef4444"},
            'steps': [
                {'range': [0,  30], 'color': '#22c55e'},
                {'range': [30, 70], 'color': '#eab308'},
                {'range': [70,100], 'color': '#ef4444'}
            ]
        }
    ))
    fig.update_layout(paper_bgcolor="rgba(0,0,0,0)", font_color="white")
    return fig

# ==============================
# 4. FORENSIC EXPLANATION ENGINE (SHAP)
# ==============================

def generate_forensic_explanation(dlq_df, probs, dlq_pred, full_df):
    explanation_text = ""

    try:
        malicious_idx = np.where(dlq_pred == 1)[0]

        if len(malicious_idx) == 0:
            return "No malicious logs detected in DLQ — no explanation needed."

        # Top 5 highest confidence malicious logs
        top_n       = min(5, len(malicious_idx))
        top_local   = malicious_idx[np.argsort(probs[malicious_idx])[::-1][:top_n]]
        top_global  = dlq_df.index[top_local]

        X_explain    = dlq_df[mlp_features].fillna(0).astype(np.float32).iloc[top_local]
        X_scaled_exp = scaler.transform(X_explain)
        X_clipped_exp= np.clip(X_scaled_exp, -10, 10)

        # KernelExplainer — model agnostic, works with any Keras model
        background  = X_clipped_exp[:min(10, len(X_clipped_exp))]
        explainer   = shap.KernelExplainer(
            lambda x: model.predict(x, verbose=0).flatten(),
            background
        )
        shap_values = explainer.shap_values(X_clipped_exp, nsamples=50)

        explanation_text += "=" * 60 + "\n"
        explanation_text += "  AI FORENSIC EXPLANATION — TOP SUSPICIOUS LOGS\n"
        explanation_text += "=" * 60 + "\n\n"

        src_col = 'IPV4_SRC_ADDR' if 'IPV4_SRC_ADDR' in full_df.columns else None
        dst_col = 'IPV4_DST_ADDR' if 'IPV4_DST_ADDR' in full_df.columns else None

        for rank, (local_i, global_i) in enumerate(zip(range(top_n), top_global)):
            prob_val = probs[top_local[rank]]
            row_data = dlq_df.loc[global_i]

            explanation_text += f"LOG #{rank+1}  —  Threat Confidence: {prob_val:.1%}\n"
            explanation_text += "-" * 50 + "\n"

            if src_col:
                explanation_text += f"  Source IP      : {full_df.loc[global_i, src_col]}\n"
            if dst_col:
                explanation_text += f"  Destination IP : {full_df.loc[global_i, dst_col]}\n"
            explanation_text += f"  Protocol       : {int(row_data.get('PROTOCOL', 0))}\n"
            explanation_text += f"  L7 Protocol    : {int(row_data.get('L7_PROTO', 0))}\n\n"

            # Top 5 SHAP contributing features
            sv            = shap_values[rank]
            contributions = sorted(
                zip(mlp_features, sv),
                key=lambda x: abs(x[1]),
                reverse=True
            )[:5]

            explanation_text += "  Top Contributing Features (SHAP):\n"
            for feat, impact in contributions:
                direction = "↑ increased" if impact > 0 else "↓ reduced"
                val       = row_data.get(feat, 0)
                explanation_text += (
                    f"    • {feat:<25} = {float(val):>10.4f}  "
                    f"→ {direction} threat  (SHAP: {impact:+.4f})\n"
                )

            # Behaviour interpretation based on top features
            top_feat_names = [f for f, _ in contributions]
            explanation_text += "\n  AI Interpretation:\n"

            if "RETRANS_RATIO_IN" in top_feat_names or "RETRANS_RATIO_OUT" in top_feat_names:
                explanation_text += "    - High retransmission ratio suggests brute force or unstable malicious connection.\n"
            if "BYTE_RATIO" in top_feat_names:
                explanation_text += "    - Asymmetric byte ratio indicates unusual inbound/outbound data imbalance.\n"
            if "PKTS_PER_MS" in top_feat_names:
                explanation_text += "    - Elevated packet rate may indicate flooding or DDoS behaviour.\n"
            if "TTL_DIFF" in top_feat_names:
                explanation_text += "    - TTL variance suggests traffic passing through unexpected network hops.\n"
            if "PKT_SIZE_RATIO" in top_feat_names:
                explanation_text += "    - Packet size anomaly detected — possible fragmentation or evasion technique.\n"
            if "FLOW_DURATION_MILLISECONDS" in top_feat_names:
                explanation_text += "    - Abnormal flow duration differs from baseline traffic patterns.\n"
            if "TOTAL_BYTES" in top_feat_names or "IN_BYTES" in top_feat_names:
                explanation_text += "    - Abnormal data volume transferred during this flow.\n"

            # Recommended response
            explanation_text += "\n  Recommended Response:\n"
            if prob_val > 0.85:
                explanation_text += "    ⚠  CRITICAL — Immediate analyst investigation required.\n"
                explanation_text += "       Isolate source IP and correlate with surrounding logs.\n"
            elif prob_val > 0.65:
                explanation_text += "    ⚡ HIGH — Correlate with surrounding telemetry.\n"
                explanation_text += "       Validate whether this matches a known attack pattern.\n"
            else:
                explanation_text += "    ℹ  MEDIUM — Monitor for persistence.\n"
                explanation_text += "       Review if this event forms part of a larger pattern.\n"

            explanation_text += "\n"

    except Exception as e:
        import traceback
        explanation_text = f"Forensic explanation error:\n{traceback.format_exc()}"

    return explanation_text

# ==============================
# 5. MAIN PIPELINE
# ==============================

def process_network_logs(file_obj):
    try:
        df = pd.read_csv(file_obj.name)
        df = preprocess_logs(df)

        for col in mlp_features + trained_features:
            if col not in df.columns:
                df[col] = 0

        # ==============================
        # STAGE 1: GATEKEEPER
        # ==============================

        X_gate       = df.reindex(columns=trained_features, fill_value=0)
        gate_scores = gatekeeper_pkg['model'].predict_proba(X_gate)[:,1]
        is_anomalous = pd.Series(
            gate_scores <= gatekeeper_pkg['threshold'], index=df.index
        )
        is_normal = ~is_anomalous

        gate_normal_count  = int(is_normal.sum())
        gate_anomaly_count = int(is_anomalous.sum())

        main_df = df[is_normal].copy()
        dlq_df  = df[is_anomalous].copy()

        # ==============================
        # STAGE 2: KAFKA ROUTING
        # ==============================

        normal_records  = main_df.to_dict(orient="records")
        anomaly_records = dlq_df.to_dict(orient="records")

        for record in normal_records:
            clean = {k: (None if pd.isna(v) else v) for k, v in record.items()}
            producer.send("main_logs", clean)
        for record in anomaly_records:
            clean = {k: (None if pd.isna(v) else v) for k, v in record.items()}
            producer.send("dlq_logs", clean)

        producer.flush()
        main_count = len(normal_records)
        dlq_count  = len(anomaly_records)

        # ==============================
        # STAGE 3: WINDOW ANALYSIS (normal/main logs)
        # Conservative thresholds to minimise false positives
        # ==============================

        def window_analysis(main_df, window_size=500):
            if len(main_df) == 0:
                return np.zeros(0, dtype=int)

            scores = np.zeros(len(main_df), dtype=int)
            v = main_df.reset_index(drop=True)

            scores += (v["RETRANS_RATIO_IN"]  > 0.7).astype(int) * 2
            scores += (v["RETRANS_RATIO_OUT"] > 0.7).astype(int) * 2
            scores += (v["BYTE_RATIO"]        > 400).astype(int) * 2
            scores += (v["PKT_SIZE_RATIO"]    > 180).astype(int)
            scores += (v["PKTS_PER_MS"]       > 15 ).astype(int)
            scores += (v["TTL_DIFF"]          > 120).astype(int)
            scores += (v["TOTAL_BYTES"]       > 5e7).astype(int)

            for start in range(0, len(v), window_size):
                window = v.iloc[start:start + window_size]
                window_suspicious = (
                    window["RETRANS_RATIO_IN"].mean()  > 0.5  or
                    window["RETRANS_RATIO_OUT"].mean() > 0.55 or
                    window["BYTE_RATIO"].mean()        > 300  or
                    window["PKT_RATIO"].mean()         > 150  or
                    window["PKT_SIZE_RATIO"].mean()    > 130
                )
                if window_suspicious:
                    scores[start:start + len(window)] += 1

            return (scores >= 5).astype(int)

        main_pred        = window_analysis(main_df)
        window_malicious = int((main_pred == 1).sum())
        window_benign    = int((main_pred == 0).sum())

        # ==============================
        # STAGE 4: DEEP AI — MLP (DLQ / anomalous logs)
        # ==============================

        use_threshold = mlp_threshold
        probs         = np.array([])

        if len(dlq_df) > 0:
            X_mlp     = dlq_df[mlp_features].fillna(0).astype(np.float32)
            X_scaled  = scaler.transform(X_mlp)
            X_clipped = np.clip(X_scaled, -10, 10)
            probs     = model.predict(X_clipped, verbose=0).flatten()

            print(f"Prob stats — min:{probs.min():.3f} max:{probs.max():.3f} "
                  f"mean:{probs.mean():.3f} median:{np.median(probs):.3f}")

            if "Label" in df.columns:
                y_dlq      = df.loc[dlq_df.index, "Label"].astype(int).values
                best_t     = 0.50
                best_score = float('inf')

                for t in np.arange(0.10, 0.96, 0.01):
                    preds    = (probs > t).astype(int)
                    fn       = int(((y_dlq == 1) & (preds == 0)).sum())
                    fp_c     = int(((y_dlq == 0) & (preds == 1)).sum())
                    combined = (fn * 2) + fp_c
                    if combined < best_score:
                        best_score = combined
                        best_t     = t

                preds_best = (probs > best_t).astype(int)
                fn_best    = int(((y_dlq == 1) & (preds_best == 0)).sum())
                fp_best    = int(((y_dlq == 0) & (preds_best == 1)).sum())
                print(f"Best threshold: {best_t:.4f}  FN={fn_best}  FP={fp_best}  "
                      f"combined={fn_best*2+fp_best}")

                np.save(f"{BASE_PATH}/threshold_dlq_tuned.npy", best_t)
                use_threshold = best_t

            else:
                tuned_path    = f"{BASE_PATH}/threshold_dlq_tuned.npy"
                use_threshold = float(np.load(tuned_path)) if os.path.exists(tuned_path) else 0.50

            dlq_pred_raw      = (probs > use_threshold).astype(int)
            low_conf_positive = (dlq_pred_raw == 1) & (probs < 0.50)
            hard_signal       = (
                (dlq_df["RETRANS_RATIO_IN"].values  > 0.5) |
                (dlq_df["RETRANS_RATIO_OUT"].values > 0.5) |
                (dlq_df["BYTE_RATIO"].values        > 300) |
                (dlq_df["PKTS_PER_MS"].values       > 10)
            )
            dlq_pred = dlq_pred_raw.copy()
            dlq_pred[low_conf_positive & ~hard_signal] = 0

            mlp_malicious = int((dlq_pred == 1).sum())
            mlp_benign    = int((dlq_pred == 0).sum())

        else:
            dlq_pred      = np.array([])
            mlp_malicious = 0
            mlp_benign    = 0

        # ==============================
        # STAGE 5: MERGE RESULTS
        # ==============================

        final_pred = np.zeros(len(df))

        if len(main_pred) > 0:
            final_pred[main_df.index] = main_pred
        if len(dlq_pred) > 0:
            final_pred[dlq_df.index] = dlq_pred

        # ==============================
        # STAGE 6: VERDICT
        # ==============================

        df["Verdict"]   = np.where(final_pred == 1, "MALICIOUS", "BENIGN")
        malicious_count = int((final_pred == 1).sum())
        benign_count    = int((final_pred == 0).sum())
        severity        = (malicious_count / len(df)) * 100
        gauge           = create_gauge(severity)

        # ==============================
        # STAGE 7: FORENSIC EXPLANATION (SHAP)
        # ==============================

        if len(dlq_df) > 0 and len(probs) > 0:
            forensic_explanation = generate_forensic_explanation(
                dlq_df, probs, dlq_pred, df
            )
        else:
            forensic_explanation = "No anomalous logs routed to DLQ — explanation not required."

        # ==============================
        # STAGE 8: FORENSIC REPORT
        # ==============================

        report = f"""
FORENSIC STREAM REPORT
============================

TOTAL LOGS
----------------------------
Total logs processed : {len(df)}

GATEKEEPER STAGE
----------------------------
Normal logs          : {gate_normal_count}
Anomalous logs       : {gate_anomaly_count}

KAFKA ROUTING
----------------------------
Sent to main_logs    : {main_count}
Sent to dlq_logs     : {dlq_count}

WINDOW ANALYSIS (Main Logs)
----------------------------
Logs analysed        : {len(main_df)}
Benign detected      : {window_benign}
Malicious detected   : {window_malicious}

DEEP AI — MLP (DLQ Logs)
----------------------------
Logs analysed        : {len(dlq_df)}
Benign detected      : {mlp_benign if len(dlq_df) > 0 else 0}
Malicious detected   : {mlp_malicious if len(dlq_df) > 0 else 0}
Decision threshold   : {use_threshold:.4f}

FINAL VERDICT
----------------------------
Total benign         : {benign_count}
Total malicious      : {malicious_count}
Threat severity      : {severity:.1f}%
"""

        if "Label" in df.columns:
            y_true = df["Label"].astype(int).values
            acc  = accuracy_score(y_true, final_pred)
            prec = precision_score(y_true, final_pred, zero_division=0)
            rec  = recall_score(y_true, final_pred, zero_division=0)
            f1   = f1_score(y_true, final_pred, zero_division=0)

            main_mask_arr = np.zeros(len(df), dtype=bool)
            dlq_mask_arr  = np.zeros(len(df), dtype=bool)
            if len(main_df) > 0:
                main_mask_arr[main_df.index] = True
            if len(dlq_df) > 0:
                dlq_mask_arr[dlq_df.index] = True

            if main_mask_arr.sum() > 0 and len(main_pred) > 0:
                w_true       = y_true[main_mask_arr]
                w_pred       = main_pred
                w_actual_atk = int(w_true.sum())
                w_acc        = accuracy_score(w_true, w_pred)

                if w_actual_atk == 0:
                    window_perf = f"""
  Logs analysed            : {len(w_true)}
  Actual attacks in bucket : {w_actual_atk}
  Accuracy                 : {w_acc:.4f}
  Precision/Recall/F1      : N/A — no attacks reached this stage
  ✓ Gatekeeper correctly isolated all anomalous traffic to DLQ"""
                else:
                    w_prec = precision_score(w_true, w_pred, zero_division=0)
                    w_rec  = recall_score(w_true, w_pred, zero_division=0)
                    w_f1   = f1_score(w_true, w_pred, zero_division=0)
                    w_cm   = confusion_matrix(w_true, w_pred)
                    window_perf = f"""
  Logs analysed            : {len(w_true)}
  Actual attacks in bucket : {w_actual_atk}
  Accuracy                 : {w_acc:.4f}
  Precision                : {w_prec:.4f}
  Recall                   : {w_rec:.4f}
  F1 Score                 : {w_f1:.4f}
  TN={w_cm[0,0]}  FP={w_cm[0,1]}  FN={w_cm[1,0]}  TP={w_cm[1,1]}"""
            else:
                window_perf = "\n  No main logs to analyse"

            if dlq_mask_arr.sum() > 0 and len(dlq_pred) > 0:
                m_true = y_true[dlq_mask_arr]
                m_pred = dlq_pred
                m_acc  = accuracy_score(m_true, m_pred)
                m_prec = precision_score(m_true, m_pred, zero_division=0)
                m_rec  = recall_score(m_true, m_pred, zero_division=0)
                m_f1   = f1_score(m_true, m_pred, zero_division=0)
                m_cm   = confusion_matrix(m_true, m_pred)
                mlp_perf = f"""
  Logs analysed            : {len(m_true)}
  Actual attacks in bucket : {int(m_true.sum())}
  Accuracy                 : {m_acc:.4f}
  Precision                : {m_prec:.4f}
  Recall                   : {m_rec:.4f}
  F1 Score                 : {m_f1:.4f}
  TN={m_cm[0,0]}  FP={m_cm[0,1]}  FN={m_cm[1,0]}  TP={m_cm[1,1]}"""
            else:
                mlp_perf = "\n  No DLQ logs to analyse"

            cm = confusion_matrix(y_true, final_pred)
            report += f"""
============================
PERFORMANCE METRICS
============================

OVERALL (full dataset)
----------------------------
  Accuracy             : {acc:.4f}
  Precision            : {prec:.4f}
  Recall               : {rec:.4f}
  F1 Score             : {f1:.4f}

  Confusion Matrix:
  Predicted →       Benign   Malicious
  Actual Benign   : {cm[0,0]:>6}   {cm[0,1]:>9}
  Actual Malicious: {cm[1,0]:>6}   {cm[1,1]:>9}

  False Positives (benign flagged as attack) : {cm[0,1]}
  False Negatives (attacks missed)           : {cm[1,0]}

WINDOW ANALYSIS (main logs){window_perf}

DEEP AI — MLP (DLQ logs){mlp_perf}
"""

        # ==============================
        # STAGE 9: DISPLAY TABLE
        # ==============================

        display_cols = ['IPV4_SRC_ADDR','IPV4_DST_ADDR','L4_DST_PORT','PROTOCOL','Verdict']
        for col in display_cols:
            if col not in df.columns:
                df[col] = "N/A"

        df_display = df[display_cols].copy()
        df_display.columns = ['Source IP','Destination','Port','Protocol','Verdict']
        df_display.insert(0, "Timestamp", pd.Timestamp.now().strftime("%H:%M:%S"))

        return df_display, df, report, gauge, forensic_explanation

    except Exception as e:
        import traceback
        return (pd.DataFrame(), pd.DataFrame(),
                f"ERROR:\n{traceback.format_exc()}",
                create_gauge(0), "Error generating explanation.")


# ==============================
# 6. CYBER UI
# ==============================

css = """
body { background: linear-gradient(135deg,#020617,#020617,#020617,#0f172a); }
.gradio-container { max-width:1400px !important; margin:auto; }
.panel {
    background:rgba(15,23,42,0.75); border-radius:18px; padding:25px;
    border:1px solid rgba(56,189,248,0.4); box-shadow:0 0 25px rgba(56,189,248,0.15);
}
h1 { font-size:38px !important; }
textarea { height:350px !important; font-family:monospace; font-size:14px; }
"""

with gr.Blocks(css=css, theme=gr.themes.Soft()) as demo:

    gr.HTML("<h1 style='text-align:center;color:#38bdf8'>🛡️ AI CYBER-FORENSIC COMMAND CENTER</h1>")

    with gr.Row():
        log_type   = gr.Radio(["NETWORK LOGS","OS LOGS","APP LOGS"],
                               value="NETWORK LOGS", label="Select Module")
        file_input = gr.File(label="Upload Log File (CSV)", file_types=[".csv"])

    with gr.Tabs():
        with gr.Tab("📡 Traffic Verdicts"):
            verdict_table = gr.Dataframe(interactive=False)
        with gr.Tab("📄 Full Network Logs"):
            full_table = gr.Dataframe(interactive=False)
        with gr.Tab("🔍 AI Forensic Report"):
            status_text = gr.Textbox(lines=20, interactive=False)
        with gr.Tab("⚠️ Threat Severity"):
            gauge_plot = gr.Plot()
        with gr.Tab("🧠 Forensic Explanation"):
            explanation_text = gr.Textbox(
                lines=35, interactive=False,
                label="SHAP-Based AI Explanation — Top 5 Most Suspicious Logs"
            )

    file_input.change(
        fn=process_network_logs,
        inputs=file_input,
        outputs=[verdict_table, full_table, status_text, gauge_plot, explanation_text]
    )

demo.launch(share=True, debug=False)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
MLP threshold loaded: 0.9507


/tmp/ipykernel_164/2455531133.py:576: DeprecationWarning:

The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.

/tmp/ipykernel_164/2455531133.py:576: DeprecationWarning:

The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.



Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://e7c6ef505b872511f9.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
# import gradio as gr
# import pandas as pd
# import numpy as np
# import tensorflow as tf
# import joblib
# import json
# import plotly.graph_objects as go
# from google.colab import drive
# from kafka import KafkaProducer
# from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
# # ==============================
# # PATTERN STORE
# # ==============================
# import hashlib, os

# PATTERN_STORE_PATH = f"{BASE_PATH}/pattern_store.json"

# def load_pattern_store():
#     if os.path.exists(PATTERN_STORE_PATH):
#         with open(PATTERN_STORE_PATH, "r") as f:
#             return json.load(f)
#     return {}

# def save_pattern_store(store):
#     with open(PATTERN_STORE_PATH, "w") as f:
#         json.dump(store, f, indent=2)

# def make_fingerprint(row):
#     # Bucket continuous values so minor variations still match
#     protocol     = int(row.get("PROTOCOL", 0))
#     l7_proto     = int(row.get("L7_PROTO", 0))
#     byte_ratio   = int(row.get("BYTE_RATIO", 0) // 10)   # bucket size 10
#     pkt_ratio    = int(row.get("PKT_RATIO", 0) // 10)
#     ttl_diff     = int(row.get("TTL_DIFF", 0) // 5)      # bucket size 5
#     retrans_in   = int(row.get("RETRANS_RATIO_IN", 0) // 0.1)
#     retrans_out  = int(row.get("RETRANS_RATIO_OUT", 0) // 0.1)

#     key = f"{protocol}|{l7_proto}|{byte_ratio}|{pkt_ratio}|{ttl_diff}|{retrans_in}|{retrans_out}"
#     return hashlib.md5(key.encode()).hexdigest()[:12]

# def update_pattern_store(store, df, verdicts):
#     """After AI classifies logs, update pattern store with results."""
#     CONFIDENCE_THRESHOLD = 10
#     for i, (_, row) in enumerate(df.iterrows()):
#         fp = make_fingerprint(row)
#         label = "ATTACK" if verdicts[i] == 1 else "BENIGN"
#         if fp not in store:
#             store[fp] = {"label": label, "count": 1, "confident": False}
#         else:
#             if store[fp]["label"] == label:
#                 store[fp]["count"] += 1
#                 if store[fp]["count"] >= CONFIDENCE_THRESHOLD:
#                     store[fp]["confident"] = True
#             else:
#                 # Conflicting label — reset, not trustworthy
#                 store[fp] = {"label": label, "count": 1, "confident": False}
#     save_pattern_store(store)
#     return store

# pattern_store = load_pattern_store()
# print(f"Pattern store loaded: {len(pattern_store)} known patterns")
# # ==============================
# # 1. INITIALIZATION
# # ==============================

# drive.mount('/content/drive')
# BASE_PATH = "/content/drive/MyDrive/Colab Notebooks"

# model          = tf.keras.models.load_model(f"{BASE_PATH}/mlp_model.keras")
# scaler         = joblib.load(f"{BASE_PATH}/scaler.pkl")
# gatekeeper_pkg = joblib.load(f"{BASE_PATH}/gatekeeper_network.pkl")

# # BUG FIX 3 — load the optimal threshold saved during training instead of hardcoding 0.5
# mlp_threshold  = float(np.load(f"{BASE_PATH}/threshold.npy"))
# print(f"MLP threshold loaded: {mlp_threshold:.4f}")

# trained_features = gatekeeper_pkg['features']

# producer = KafkaProducer(
#     bootstrap_servers="kafka-24f52108-loginvestigation.i.aivencloud.com:27515",
#     security_protocol="SSL",
#     ssl_cafile=f"{BASE_PATH}/ca_b.pem",
#     ssl_certfile=f"{BASE_PATH}/service_b.cert",
#     ssl_keyfile=f"{BASE_PATH}/service_b.key",
#     value_serializer=lambda v: json.dumps(v, default=str).encode("utf-8"),
#     batch_size=131072,
#     linger_ms=100,
#     compression_type="gzip",
#     acks=1
# )

# mlp_features = [
#     "IN_BYTES","OUT_BYTES","IN_PKTS","OUT_PKTS",
#     "FLOW_DURATION_MILLISECONDS",
#     "LONGEST_FLOW_PKT","SHORTEST_FLOW_PKT",
#     "TCP_FLAGS","CLIENT_TCP_FLAGS","SERVER_TCP_FLAGS",
#     "RETRANSMITTED_IN_PKTS","RETRANSMITTED_OUT_PKTS",
#     "MIN_TTL","MAX_TTL","PROTOCOL","L7_PROTO",
#     "BYTE_RATIO","PKT_RATIO",
#     "BYTES_PER_PKT_IN","BYTES_PER_PKT_OUT",
#     "TOTAL_BYTES","TOTAL_PKTS",
#     "BYTES_PER_MS","PKTS_PER_MS",
#     "PKT_SIZE_RATIO",
#     "RETRANS_RATIO_IN","RETRANS_RATIO_OUT",
#     "TTL_DIFF"
# ]

# # ==============================
# # 2. PREPROCESSING PIPELINE
# # ==============================
# def preprocess_logs(df):
#     df = df.copy()
#     df = df.fillna(0)
#     eps = 1e-9

#     df["FLOW_START_MILLISECONDS"] = pd.to_datetime(
#         df["FLOW_START_MILLISECONDS"], unit="ms", errors="coerce"
#     )
#     df["START_HOUR"] = df["FLOW_START_MILLISECONDS"].dt.hour
#     df["hour_sin"]   = np.sin(2 * np.pi * df["START_HOUR"] / 24)
#     df["hour_cos"]   = np.cos(2 * np.pi * df["START_HOUR"] / 24)

#     df["BYTE_RATIO"]        = df["IN_BYTES"]  / (df["OUT_BYTES"]  + eps)
#     df["PKT_RATIO"]         = df["IN_PKTS"]   / (df["OUT_PKTS"]   + eps)
#     df["BYTES_PER_PKT_IN"]  = df["IN_BYTES"]  / (df["IN_PKTS"]    + eps)
#     df["BYTES_PER_PKT_OUT"] = df["OUT_BYTES"] / (df["OUT_PKTS"]   + eps)
#     df["TOTAL_BYTES"]       = df["IN_BYTES"]  +  df["OUT_BYTES"]
#     df["TOTAL_PKTS"]        = df["IN_PKTS"]   +  df["OUT_PKTS"]
#     df["BYTES_PER_MS"]      = df["TOTAL_BYTES"] / (df["FLOW_DURATION_MILLISECONDS"] + eps)
#     df["PKTS_PER_MS"]       = df["TOTAL_PKTS"]  / (df["FLOW_DURATION_MILLISECONDS"] + eps)
#     df["PKT_SIZE_RATIO"]    = df["LONGEST_FLOW_PKT"] / (df["SHORTEST_FLOW_PKT"] + eps)
#     df["RETRANS_RATIO_IN"]  = df["RETRANSMITTED_IN_PKTS"]  / (df["IN_PKTS"]  + eps)
#     df["RETRANS_RATIO_OUT"] = df["RETRANSMITTED_OUT_PKTS"] / (df["OUT_PKTS"] + eps)
#     df["TTL_DIFF"]          = df["MAX_TTL"] - df["MIN_TTL"]
#     df["BYTE_RATIO"]     = df["BYTE_RATIO"].clip(0, 500)
#     df["PKT_RATIO"]      = df["PKT_RATIO"].clip(0, 500)
#     df["PKT_SIZE_RATIO"] = df["PKT_SIZE_RATIO"].clip(0, 200)
#     df = df.drop(columns=[
#         "FLOW_START_MILLISECONDS", "FLOW_END_MILLISECONDS", "START_HOUR"
#     ], errors="ignore")

#     return df

# # ==============================
# # 3. FORENSIC ENGINE
# # ==============================

# def create_gauge(score):
#     fig = go.Figure(go.Indicator(
#         mode="gauge+number",
#         value=score,
#         title={'text': "Threat Level"},
#         gauge={
#             'axis': {'range': [0, 100]},
#             'bar': {'color': "#ef4444"},
#             'steps': [
#                 {'range': [0,  30], 'color': '#22c55e'},
#                 {'range': [30, 70], 'color': '#eab308'},
#                 {'range': [70,100], 'color': '#ef4444'}
#             ]
#         }
#     ))
#     fig.update_layout(paper_bgcolor="rgba(0,0,0,0)", font_color="white")
#     return fig


# def process_network_logs(file_obj):
#     try:
#         df = pd.read_csv(file_obj.name)
#         df = preprocess_logs(df)

#         for col in mlp_features + trained_features:
#             if col not in df.columns:
#                 df[col] = 0
#         # ==============================
# # STAGE 0: PATTERN LOOKUP
# # ==============================
# global pattern_store

# pattern_hits_benign  = 0
# pattern_hits_attack  = 0
# pattern_unknown_mask = []

# stage0_pred = np.full(len(df), -1)  # -1 = unknown, 0 = benign, 1 = attack

# for i, (_, row) in enumerate(df.iterrows()):
#     fp = make_fingerprint(row)
#     if fp in pattern_store and pattern_store[fp]["confident"]:
#         if pattern_store[fp]["label"] == "ATTACK":
#             stage0_pred[i] = 1
#             pattern_hits_attack += 1
#         else:
#             stage0_pred[i] = 0
#             pattern_hits_benign += 1

# # Only unknown logs go to gatekeeper
# unknown_mask = stage0_pred == -1
# known_mask   = stage0_pred != -1
# df_unknown   = df[unknown_mask].copy()
# df_known     = df[known_mask].copy()

# pattern_hits_total = pattern_hits_benign + pattern_hits_attack
# print(f"Pattern store: {pattern_hits_total} hits ({pattern_hits_benign} benign, {pattern_hits_attack} attack), {unknown_mask.sum()} unknown → going to AI")
#         # ==============================
#         # 1. GATEKEEPER
#         # ==============================

#         X_gate     = df.reindex(columns=trained_features, fill_value=0)
#         gate_scores = gatekeeper_pkg['model'].decision_function(X_gate)

#         # BUG FIX 1 — gatekeeper is an anomaly detector (e.g. IsolationForest / OneClassSVM)
#         # Higher score = more normal. Anomalies score BELOW the threshold.
#         # Original code had this inverted, sending everything to main and starving the MLP.
#         is_anomalous = pd.Series(gate_scores <= gatekeeper_pkg['threshold'], index=df.index)
#   # True = suspicious → DLQ → MLP
#         is_normal    = ~is_anomalous
#         gate_normal_count = int(is_normal.sum())
#         gate_anomaly_count = int(is_anomalous.sum())                               # True = routine    → main → window analysis

#         main_df = df[is_normal].copy()
#         dlq_df  = df[is_anomalous].copy()

#         # ==============================
#         # 2. SEND TO KAFKA
#         # ==============================

#         main_count = 0
#         dlq_count  = 0

#         # for i, row in df.iterrows():
#         #     record    = row.to_dict()
#         #     clean_msg = {k: (None if pd.isna(v) else v) for k, v in record.items()}
#         #     if is_normal.loc[i]:
#         #         producer.send("main_logs", clean_msg)
#         #         main_count += 1
#         #     else:
#         #         producer.send("dlq_logs", clean_msg)
#         #         dlq_count += 1

#         # producer.flush()
#         normal_records  = df[is_normal].to_dict(orient="records")
#         anomaly_records = df[is_anomalous].to_dict(orient="records")

#         for record in normal_records:
#             clean = {k: (None if pd.isna(v) else v) for k, v in record.items()}
#             producer.send("main_logs", clean)
#         for record in anomaly_records:
#             clean = {k: (None if pd.isna(v) else v) for k, v in record.items()}
#             producer.send("dlq_logs", clean)

#         producer.flush()
#         main_count = len(normal_records)
#         dlq_count  = len(anomaly_records)

#         # ==============================
#         # 3. WINDOW ANALYSIS (NORMAL/MAIN LOGS)
#         # BUG FIX 2 — replaced hardcoded byte/retrans thresholds with
#         # ratio-based rules that generalise across networks
#         # ==============================
#         # def window_analysis(main_df, window_size=500):
#         #   results = np.zeros(len(main_df), dtype=int)
#         #   if len(main_df) == 0:
#         #     return results
#         #   for start in range(0, len(main_df), window_size):
#         #     window = main_df.iloc[start:start + window_size]
#         #     idx    = window.index
#         #     # --- Window-level context (aggregate signals) ---
#         #     avg_retrans_in  = window["RETRANS_RATIO_IN"].mean()
#         #     avg_retrans_out = window["RETRANS_RATIO_OUT"].mean()
#         #     avg_byte_ratio  = window["BYTE_RATIO"].mean()
#         #     avg_pkt_ratio   = window["PKT_RATIO"].mean()
#         #     avg_pkt_size_r  = window["PKT_SIZE_RATIO"].mean()
#         #     window_suspicious = (
#         #     avg_retrans_in  > 0.38 or
#         #     avg_retrans_out > 0.42 or
#         #     avg_byte_ratio  > 200  or
#         #     avg_pkt_ratio   > 100  or
#         #     avg_pkt_size_r  > 100
#         #     )
#         #     # --- Per-row classification within the window ---
#         #     for i, (_, row) in enumerate(window.iterrows()):
#         #       row_score = 0
#         #       #Strong individual signals
#         #       if row["RETRANS_RATIO_IN"]  > 0.5:  row_score += 2
#         #       if row["RETRANS_RATIO_OUT"] > 0.5:  row_score += 2
#         #       if row["BYTE_RATIO"]        > 300:  row_score += 2
#         #       if row["PKT_SIZE_RATIO"]    > 150:  row_score += 1
#         #       if row["PKTS_PER_MS"]       > 10:   row_score += 1
#         #       if row["TTL_DIFF"]          > 100:  row_score += 1
#         #       if row["TOTAL_BYTES"]       > 1e7:  row_score += 1
#         #       # Boost score if window context is already suspicious
#         #       if window_suspicious:
#         #         row_score += 1
#         #       results[i + start] = 1 if row_score >= 3 else 0
#         #   return results
#         def window_analysis(main_df, window_size=500):
#             if len(main_df) == 0:
#                 return np.zeros(0, dtype=int)

#             # Vectorized scoring — no inner loop
#             scores = np.zeros(len(main_df), dtype=int)
#             v = main_df.reset_index(drop=True)  # clean integer index

#             scores += (v["RETRANS_RATIO_IN"]  > 0.5).astype(int) * 2
#             scores += (v["RETRANS_RATIO_OUT"] > 0.5).astype(int) * 2
#             scores += (v["BYTE_RATIO"]        > 300).astype(int) * 2
#             scores += (v["PKT_SIZE_RATIO"]    > 150).astype(int)
#             scores += (v["PKTS_PER_MS"]       > 10 ).astype(int)
#             scores += (v["TTL_DIFF"]          > 100).astype(int)
#             scores += (v["TOTAL_BYTES"]       > 1e7).astype(int)

#             # Window context boost — vectorized per window
#             for start in range(0, len(v), window_size):
#                 window = v.iloc[start:start + window_size]
#                 window_suspicious = (
#                     window["RETRANS_RATIO_IN"].mean()  > 0.38 or
#                     window["RETRANS_RATIO_OUT"].mean() > 0.42 or
#                     window["BYTE_RATIO"].mean()        > 200  or
#                     window["PKT_RATIO"].mean()         > 100  or
#                     window["PKT_SIZE_RATIO"].mean()    > 100
#                 )
#                 if window_suspicious:
#                     scores[start:start + len(window)] += 1

#             return (scores >= 3).astype(int)
#         main_pred = window_analysis(main_df)
#         window_malicious = int((main_pred == 1).sum())   # ← ADD THIS
#         window_benign    = int((main_pred == 0).sum())

#         # ==============================
#         # 4. DEEP AI (DLQ / ANOMALOUS LOGS)
#         # # ==============================
#         # # Force re-search — delete old tuned threshold so fresh search always runs
#         # import os
#         # tuned_path = f"{BASE_PATH}/threshold_dlq_tuned.npy"
#         # if os.path.exists(tuned_path):
#         #     os.remove(tuned_path)
#         #     print("Old threshold deleted — running fresh search...")

#         use_threshold = mlp_threshold

#         if len(dlq_df) > 0:
#             X_mlp     = dlq_df[mlp_features].fillna(0).astype(np.float32)
#             X_scaled  = scaler.transform(X_mlp)
#             X_clipped = np.clip(X_scaled, -10, 10)
#             probs     = model.predict(X_clipped, verbose=0).flatten()

#             # Print score distribution so we can see where attacks cluster
#             print(f"Prob stats — min:{probs.min():.3f} max:{probs.max():.3f} "
#                   f"mean:{probs.mean():.3f} median:{np.median(probs):.3f}")

#             if "Label" in df.columns:
#                 y_dlq    = df.loc[dlq_df.index, "Label"].astype(int).values
#                 best_t   = 0.50
#                 best_score = float('inf')

#                 for t in np.arange(0.10, 0.96, 0.01):
#                     preds = (probs > t).astype(int)

#                     fn = int(((y_dlq == 1) & (preds == 0)).sum())  # missed attacks
#                     fp = int(((y_dlq == 0) & (preds == 1)).sum())  # false alarms

#                     # Weight FN slightly higher — missing an attack is worse than a false alarm
#                     combined = (fn * 2) + fp

#                     if combined < best_score:
#                         best_score = combined
#                         best_t     = t

#                 # Print details so you can see the tradeoff
#                 preds_best = (probs > best_t).astype(int)
#                 fn_best = int(((y_dlq == 1) & (preds_best == 0)).sum())
#                 fp_best = int(((y_dlq == 0) & (preds_best == 1)).sum())
#                 print(f"Best threshold: {best_t:.4f}  FN={fn_best}  FP={fp_best}  combined={fn_best*2+fp_best}")

#                 np.save(f"{BASE_PATH}/threshold_dlq_tuned.npy", best_t)
#                 use_threshold = best_t

#             else:
#                 tuned_path = f"{BASE_PATH}/threshold_dlq_tuned.npy"
#                 use_threshold = float(np.load(tuned_path)) if os.path.exists(tuned_path) else 0.50

#             # Primary MLP prediction
#             dlq_pred_raw = (probs > use_threshold).astype(int)

#             # Secondary check — VECTORIZED
#             low_conf_positive = (dlq_pred_raw == 1) & (probs < 0.50)
#             hard_signal = (
#                 (dlq_df["RETRANS_RATIO_IN"].values  > 0.5) |
#                 (dlq_df["RETRANS_RATIO_OUT"].values > 0.5) |
#                 (dlq_df["BYTE_RATIO"].values        > 300) |
#                 (dlq_df["PKTS_PER_MS"].values       > 10)
#             )
#             dlq_pred = dlq_pred_raw.copy()
#             dlq_pred[low_conf_positive & ~hard_signal] = 0

#             mlp_malicious = int((dlq_pred == 1).sum())
#             mlp_benign    = int((dlq_pred == 0).sum())

#         else:
#             dlq_pred      = np.array([])
#             mlp_malicious = 0
#             mlp_benign    = 0
#         # ==============================
#         # 5. MERGE RESULTS
#         # ==============================

#         final_pred = np.zeros(len(df))

#         if len(main_pred) > 0:
#             final_pred[main_df.index] = main_pred

#         if len(dlq_pred) > 0:
#             final_pred[dlq_df.index] = dlq_pred

#         # ==============================
#         # 6. VERDICT
#         # ==============================

#         df["Verdict"]     = np.where(final_pred == 1, "MALICIOUS", "BENIGN")
#         malicious_count   = int((final_pred == 1).sum())
#         benign_count = int((final_pred == 0).sum())
#         severity          = (malicious_count / len(df)) * 100
#         gauge             = create_gauge(severity)


# # Replace the entire section 7 in process_network_logs:

#         # ==============================
#         # 7. FORENSIC REPORT
#         # ==============================

#         report = f"""
# FORENSIC STREAM REPORT
# ============================

# TOTAL LOGS
# ----------------------------
# Total logs processed : {len(df)}

# GATEKEEPER STAGE
# ----------------------------
# Normal logs          : {gate_normal_count}
# Anomalous logs       : {gate_anomaly_count}

# KAFKA ROUTING
# ----------------------------
# Sent to main_logs    : {main_count}
# Sent to dlq_logs     : {dlq_count}

# WINDOW ANALYSIS (Main Logs)
# ----------------------------
# Logs analysed        : {len(main_df)}
# Benign detected      : {window_benign}
# Malicious detected   : {window_malicious}

# DEEP AI — MLP (DLQ Logs)
# ----------------------------
# Logs analysed        : {len(dlq_df)}
# Benign detected      : {mlp_benign if len(dlq_df) > 0 else 0}
# Malicious detected   : {mlp_malicious if len(dlq_df) > 0 else 0}
# Decision threshold   : {use_threshold:.4f}

# FINAL VERDICT
# ----------------------------
# Total benign         : {benign_count}
# Total malicious      : {malicious_count}
# Threat severity      : {severity:.1f}%
# """

#         if "Label" in df.columns:
#             y_true = df["Label"].astype(int).values

#             acc  = accuracy_score(y_true, final_pred)
#             prec = precision_score(y_true, final_pred, zero_division=0)
#             rec  = recall_score(y_true, final_pred, zero_division=0)
#             f1   = f1_score(y_true, final_pred, zero_division=0)

#             # Per-stage breakdown (gatekeeper routing vs actual labels)
#             main_mask = is_normal.values
#             dlq_mask  = is_anomalous.values

#             if main_mask.sum() > 0 and len(main_pred) > 0:
#                 w_true         = y_true[main_mask]
#                 w_pred         = main_pred
#                 w_actual_atk   = int(w_true.sum())
#                 w_acc          = accuracy_score(w_true, w_pred)

#                 if w_actual_atk == 0:
#                     # No attacks reached window analysis — gatekeeper working perfectly
#                     window_perf = f"""
#   Logs analysed        : {len(w_true)}
#   Actual attacks in bucket : {w_actual_atk}
#   Accuracy             : {w_acc:.4f}
#   Precision/Recall/F1  : N/A — no attacks reached this stage
#   ✓ Gatekeeper correctly isolated all anomalous traffic to DLQ"""
#                 else:
#                     w_prec = precision_score(w_true, w_pred, zero_division=0)
#                     w_rec  = recall_score(w_true, w_pred, zero_division=0)
#                     w_f1   = f1_score(w_true, w_pred, zero_division=0)
#                     w_cm   = confusion_matrix(w_true, w_pred)
#                     window_perf = f"""
#   Logs analysed        : {len(w_true)}
#   Actual attacks in bucket : {w_actual_atk}
#   Accuracy             : {w_acc:.4f}
#   Precision            : {w_prec:.4f}
#   Recall               : {w_rec:.4f}
#   F1 Score             : {w_f1:.4f}
#   TN={w_cm[0,0]}  FP={w_cm[0,1]}  FN={w_cm[1,0]}  TP={w_cm[1,1]}"""
#             else:
#                 window_perf = "\n  No main logs to analyse"

#             # MLP performance
#             if dlq_mask.sum() > 0 and len(dlq_pred) > 0:
#                 m_true = y_true[dlq_mask]
#                 m_pred = dlq_pred
#                 m_acc  = accuracy_score(m_true, m_pred)
#                 m_prec = precision_score(m_true, m_pred, zero_division=0)
#                 m_rec  = recall_score(m_true, m_pred, zero_division=0)
#                 m_f1   = f1_score(m_true, m_pred, zero_division=0)
#                 m_cm   = confusion_matrix(m_true, m_pred)
#                 mlp_perf = f"""
#   Logs analysed        : {len(m_true)}
#   Actual attacks in bucket : {int(m_true.sum())}
#   Accuracy             : {m_acc:.4f}
#   Precision            : {m_prec:.4f}
#   Recall               : {m_rec:.4f}
#   F1 Score             : {m_f1:.4f}
#   TN={m_cm[0,0]}  FP={m_cm[0,1]}  FN={m_cm[1,0]}  TP={m_cm[1,1]}"""
#             else:
#                 mlp_perf = "\n  No DLQ logs to analyse"

#             cm = confusion_matrix(y_true, final_pred)

#             report += f"""
# ============================
# PERFORMANCE METRICS
# ============================

# OVERALL (full dataset)
# ----------------------------
#   Accuracy             : {acc:.4f}
#   Precision            : {prec:.4f}
#   Recall               : {rec:.4f}
#   F1 Score             : {f1:.4f}

#   Confusion Matrix:
#   Predicted →       Benign   Malicious
#   Actual Benign   : {cm[0,0]:>6}   {cm[0,1]:>9}
#   Actual Malicious: {cm[1,0]:>6}   {cm[1,1]:>9}

#   False Positives (benign flagged as attack) : {cm[0,1]}
#   False Negatives (attacks missed)           : {cm[1,0]}

# WINDOW ANALYSIS (main logs){window_perf}

# DEEP AI — MLP (DLQ logs){mlp_perf}
# """

#         # ==============================
#         # 8. DISPLAY TABLE
#         # ==============================

#         display_cols = ['IPV4_SRC_ADDR','IPV4_DST_ADDR','L4_DST_PORT','PROTOCOL','Verdict']
#         for col in display_cols:
#             if col not in df.columns:
#                 df[col] = "N/A"

#         df_display = df[display_cols].copy()
#         df_display.columns = ['Source IP','Destination','Port','Protocol','Verdict']
#         df_display.insert(0, "Timestamp", pd.Timestamp.now().strftime("%H:%M:%S"))

#         return df_display, df, report, gauge

#     except Exception as e:
#         import traceback
#         return pd.DataFrame(), pd.DataFrame(), f"ERROR:\n{traceback.format_exc()}", create_gauge(0)


# # ==============================
# # 4. CYBER UI
# # ==============================

# css = """
# body { background: linear-gradient(135deg,#020617,#020617,#020617,#0f172a); }
# .gradio-container { max-width:1400px !important; margin:auto; }
# .panel {
#     background:rgba(15,23,42,0.75); border-radius:18px; padding:25px;
#     border:1px solid rgba(56,189,248,0.4); box-shadow:0 0 25px rgba(56,189,248,0.15);
# }
# h1 { font-size:38px !important; }
# textarea { height:350px !important; font-family:monospace; font-size:14px; }
# """

# with gr.Blocks(css=css, theme=gr.themes.Soft()) as demo:

#     gr.HTML("<h1 style='text-align:center;color:#38bdf8'>🛡️ AI CYBER-FORENSIC COMMAND CENTER</h1>")

#     with gr.Row():
#         log_type   = gr.Radio(["NETWORK LOGS","OS LOGS","APP LOGS"],
#                                value="NETWORK LOGS", label="Select Module")
#         file_input = gr.File(label="Upload Log File (CSV)", file_types=[".csv"])

#     with gr.Tabs():
#         with gr.Tab("📡 Traffic Verdicts"):
#             verdict_table = gr.Dataframe(interactive=False)
#         with gr.Tab("📄 Full Network Logs"):
#             full_table = gr.Dataframe(interactive=False)
#         with gr.Tab("🔍 AI Forensic Report"):
#             status_text = gr.Textbox(lines=20, interactive=False)
#         with gr.Tab("⚠️ Threat Severity"):
#             gauge_plot = gr.Plot()

#     file_input.change(
#         fn=process_network_logs,
#         inputs=file_input,
#         outputs=[verdict_table, full_table, status_text, gauge_plot]
#     )

# demo.launch(share=True, debug=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
MLP threshold loaded: 0.9505


/tmp/ipykernel_231/3748806536.py:511: DeprecationWarning:

The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.

/tmp/ipykernel_231/3748806536.py:511: DeprecationWarning:

The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.



Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://b694d980454d313006.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


ERROR:    Exception in ASGI application
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/protocols/http/httptools_impl.py", line 416, in run_asgi
    result = await app(  # type: ignore[func-returns-value]
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/middleware/proxy_headers.py", line 60, in __call__
    return await self.app(scope, receive, send)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/fastapi/applications.py", line 1160, in __call__
    await super().__call__(scope, receive, send)
  File "/usr/local/lib/python3.12/dist-packages/starlette/applications.py", line 107, in __call__
    await self.middleware_stack(scope, receive, send)
  File "/usr/local/lib/python3.12/dist-packages/starlette/middleware/errors.py", line 186, in __call__
    raise exc
  File "/usr/local/lib/python3.12/dist-packages/starlette/middleware/error

Threshold tuned: 0.1400  FN=3  FP=599
Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://b694d980454d313006.gradio.live
